# Transformer Block: Putting the Pieces Together Lab

Build one reusable pre-norm causal Transformer block, then inspect its named boundaries. The checks separate shape, causality, gradient, and head-splitting contracts.


In [ ]:
import math
import torch
from torch import nn

torch.manual_seed(13)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd=12, n_head=3):
        super().__init__()
        if n_embd % n_head:
            raise ValueError("n_embd must be divisible by n_head")
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        def heads(t):
            return t.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k, v = heads(q), heads(k), heads(v)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)
        weights = torch.softmax(scores.masked_fill(mask, float("-inf")), dim=-1)
        y = (weights @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y), weights

class TransformerBlock(nn.Module):
    def __init__(self, n_embd=12, n_head=3):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ff = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        update, weights = self.attn(self.ln1(x))
        x = x + update
        x = x + self.ff(self.ln2(x))
        return x, weights

x = torch.randn(2, 4, 12)
block = TransformerBlock()
out, weights = block(x)
future = torch.triu(torch.ones(4, 4, dtype=torch.bool), diagonal=1)
assert out.shape == x.shape
assert torch.isfinite(out).all()
assert weights.masked_select(future).abs().sum().item() == 0.0
print("PASS: l05-13 block composition and causal checks")


In [ ]:
x = torch.randn(2, 4, 12, requires_grad=True)
norm1 = block.ln1(x)
attn_update, weights = block.attn(norm1)
after_attn = x + attn_update
norm2 = block.ln2(after_attn)
mlp_update = block.ff(norm2)
out = after_attn + mlp_update

boundaries = {
    "input": x,
    "norm1": norm1,
    "attention_update": attn_update,
    "after_attention_residual": after_attn,
    "norm2": norm2,
    "mlp_update": mlp_update,
    "output": out,
}
for name, tensor in boundaries.items():
    print(f"{name:26s} {tuple(tensor.shape)}")
    assert tuple(tensor.shape) == (2, 4, 12)

out.square().mean().backward()
assert x.grad is not None and torch.isfinite(x.grad).all()

try:
    CausalSelfAttention(n_embd=10, n_head=3)
    raise AssertionError("expected divisibility failure")
except ValueError:
    pass

print("PASS: l05-13 boundary, gradient, and head-split checks")
